# Bengali STS — Q1 Suite v2 (Human Gold Set · Encoder Adaptation · Morphology-Aware Pooling)
### Multi-seed · expanded zero-shot baselines · **adapted encoders** · **morphology-aware pooling** · prior/morph ablation · error analysis · publication figures · datasheet

This notebook produces the **complete empirical material** for a Q1 STS paper built around your
295-pair human-annotated Bengali gold set (Krippendorff α = 0.863, ICC(2,k) = 0.950).

**What changed vs v1 (and why it is now Q1-defensible)**
1. **Literature check is settled.** A prior Bengali STS set exists (Shajalal & Aono, 2018) and meets
   SemEval criteria, so "first Bengali STS dataset" is *false*. But it is **not publicly available**
   (per the BanglaBERT authors), has **< 1000 pairs**, and Bengali is **absent from SemRel2024 and
   MUSTS**. The defensible claim is therefore: *the first publicly available, natively sourced,
   human-annotated Bengali STS evaluation set with reported inter-annotator agreement.*
2. **Option 1 — encoder adaptation (the working best system).** Light contrastive fine-tuning of a
   strong multilingual encoder (LaBSE; BGE-M3 optional) on the translated STS-B pairs, evaluated on
   the native gold set. This is the realistic "best system" rather than fighting to make the
   BanglaBERT bi-encoder competitive.
3. **Option 2 — morphology-aware pooling (the novel method).** Bengali is highly inflectional;
   subword tokenisation fragments words unevenly. The new head pools **subwords → words** first, then
   applies **word-level attention**, optionally **primed by a word-level POS weight** (POS is a word
   property — this is where the prior belongs). This reframes v1's token-level primed attention,
   which tied/loses, into a principled form and lets the ablation ask a clean question: does
   word-level pooling beat token-level, and does a word-level POS prior help?
4. **Statistics fixed.** Point estimates are mean ± std over seeds; **CIs and Williams tests now use
   a consistent basis** (per-seed, averaged) instead of v1's seed-0-only mix that flipped signs.
   The "best system" is now selected by score, not hard-coded.

> **Honest scope.** This generates results and figures; you still write the paper. And no method here
> is guaranteed to beat zero-shot BGE-M3 (~90) — if morphology-aware pooling or adaptation does not
> win, that is reported plainly and is itself publishable. The Q1 spine is the **benchmark + the
> zero-shot-vs-fine-tuned landscape**; the morphology head is the modelling contribution.

> **Runtime:** GPU required (A100 ideal). 6 BanglaBERT heads × 3 seeds + 2 ablation modes × 3 +
> LaBSE adaptation × 3 ≈ **3–4 h**. Reduce `SEEDS` or comment out heads to shorten.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# CELL 1 — Install
# ============================================================
!pip install -q "transformers>=4.40" datasets sentence-transformers
!pip install -q bnlp-toolkit sacremoses sentencepiece
!pip install -q scikit-learn scipy pandas matplotlib seaborn
!pip install -q git+https://github.com/csebuetnlp/normalizer
print("\nIf you hit a NumPy/!pip resolver warning: Runtime > Restart session, then re-run from Cell 2.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 118.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.3/168.3 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 63.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 18.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 7.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ERROR: pip's dependency resolver does not currently take into account all the pack

In [ ]:
# ============================================================
# CELL 2 — Imports, device
# ============================================================
import os, json, math, pickle, random, warnings, re
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy import stats
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| torch", torch.__version__)
if DEVICE=="cpu": print("⚠️  No GPU — fine-tuning will be impractically slow. Use a GPU runtime.")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

Device: cuda | torch 2.11.0+cu128


In [ ]:
# ============================================================
# CELL 3 — Drive, paths, experiment config
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/Similarity_V2_Patched/'
FIGDIR  = PROJECT + 'figures/'
os.makedirs(FIGDIR, exist_ok=True)

PATHS = {
    'gold_test'     : '/content/drive/MyDrive/Genu/Output/bn_sts_gold_test.csv',
    'bn_stsb_cache' : PROJECT+'bn_stsb_translated.csv',
    'idf_cache'     : PROJECT+'bengali_idf.pkl',
    'tokenized'     : PROJECT+'bn_stsb_tokenized_q1v2.pt',   # bumped: now stores word-ids too
    'ckpt_dir'      : PROJECT+'checkpoints/',
    'results_csv'   : PROJECT+'q1_results.csv',
    'ablation_csv'  : PROJECT+'q1_ablation.csv',
    'erroran_csv'   : PROJECT+'q1_error_analysis.csv',
    'datasheet'     : PROJECT+'q1_datasheet.md',
    'corpus'        : ['/content/drive/MyDrive/Genu/anandabazar_articles.txt', '/content/drive/MyDrive/Genu/zeenews_articles.txt'],
}
os.makedirs(PATHS['ckpt_dir'], exist_ok=True)

BACKBONE = "csebuetnlp/banglabert"
MAX_LEN  = 64
SEEDS    = [42, 1, 2]          # >=3 for Q1; reduce to [42] for a quick dry run
EPOCHS   = 4

# --- Option 1 (encoder adaptation) config ---
ADAPT_MODELS = {"LaBSE": "sentence-transformers/LaBSE"}   # add "BGE-M3":"BAAI/bge-m3" for the strongest (heavy)
ADAPT_EPOCHS = 3
ADAPT_BS     = 32
ADAPT_LR     = 2e-5

RUN_LASER    = False
RUN_FASTTEXT = False
assert os.path.exists(PATHS['gold_test']), f"Gold file not found: {PATHS['gold_test']} — upload bn_sts_gold_test.csv to /content or the Similarity Drive folder."
plt.rcParams.update({'figure.dpi':120,'savefig.dpi':300,'font.size':11,'axes.grid':True,
                     'grid.alpha':0.3,'axes.axisbelow':True})

# --- Bengali-capable font for figures (so word-attention / Bengali tick labels render, not boxes) ---
import matplotlib.font_manager as fm
_BN_TTF = '/content/NotoSansBengali-Regular.ttf'
try:
    if not os.path.exists(_BN_TTF):
        import urllib.request
        urllib.request.urlretrieve(
            'https://raw.githubusercontent.com/googlefonts/noto-fonts/main/hinted/ttf/NotoSansBengali/NotoSansBengali-Regular.ttf',
            _BN_TTF)
    fm.fontManager.addfont(_BN_TTF)
    _BN_NAME = fm.FontProperties(fname=_BN_TTF).get_name()
    plt.rcParams['font.family'] = [_BN_NAME, 'DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False
    print(f"\u2705 Bengali font registered: {_BN_NAME}")
except Exception as _e:
    print("\u26a0\ufe0f  Bengali font NOT registered (Bengali text in figures may show as boxes):", str(_e)[:120])

print("seeds:", SEEDS, "| gold:", PATHS['gold_test'])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Bengali font registered: Noto Sans Bengali
seeds: [42, 1, 2] | gold: /content/drive/MyDrive/Genu/Output/bn_sts_gold_test.csv


In [ ]:
# ============================================================
# CELL 4 — Prepare human test set + build POS×IDF from native corpus
# ============================================================
g_raw = pd.read_csv(PATHS['gold_test'], encoding='utf-8-sig')
human = g_raw[['sentence1','sentence2','gold_score']].rename(columns={'gold_score':'score'})
print(f"✅ gold set: {len(human)} pairs (score {human.score.min():.1f}-{human.score.max():.1f})")

if not os.path.exists(PATHS['idf_cache']) and all(os.path.exists(p) for p in PATHS['corpus']):
    from normalizer import normalize
    from collections import Counter
    docs=[]
    for f in PATHS['corpus']:
        for a in json.load(open(f,encoding='utf-8'))['articles']:
            b=a.get('body','').strip()
            if b: docs.append(normalize(b))
    N=len(docs); df=Counter()
    for d in docs: df.update(set(d.split()))
    IDF_B={w: math.log(N/(1+c))+1.0 for w,c in df.items()}
    pickle.dump(IDF_B, open(PATHS['idf_cache'],'wb'))
    print(f"✅ IDF over {N} docs, {len(IDF_B):,} tokens")
elif os.path.exists(PATHS['idf_cache']):
    print("✅ IDF cache present.")
else:
    print("⚠️  No corpus -> primed/IDF priors will be POS-only.")

✅ gold set: 295 pairs (score 0.0-5.0)
✅ IDF over 7383 docs, 120,642 tokens


In [ ]:
# ============================================================
# CELL 5 — STS-B -> Bengali (NLLB, cached) for train/dev
# ============================================================
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
sts = load_dataset("mteb/stsbenchmark-sts")
def translate_stsb():
    if os.path.exists(PATHS['bn_stsb_cache']):
        d=pd.read_csv(PATHS['bn_stsb_cache']); print(f"✅ cached translation ({len(d)})"); return d
    mt_name="facebook/nllb-200-distilled-600M"
    tok=AutoTokenizer.from_pretrained(mt_name,src_lang="eng_Latn")
    mt=AutoModelForSeq2SeqLM.from_pretrained(mt_name).to(DEVICE).eval()
    bos=tok.convert_tokens_to_ids("ben_Beng")
    @torch.no_grad()
    def tr(texts,bs=32):
        out=[]
        for i in range(0,len(texts),bs):
            enc=tok([str(t) for t in texts[i:i+bs]],return_tensors="pt",padding=True,truncation=True,max_length=128).to(DEVICE)
            out+=tok.batch_decode(mt.generate(**enc,forced_bos_token_id=bos,max_length=128),skip_special_tokens=True)
            print(f"  {min(i+bs,len(texts))}/{len(texts)}",end='\r')
        print(); return out
    rows=[]
    for split in ['train','validation']:
        d=sts[split]; print(f"Translating {split} ({len(d)})...")
        s1=tr(d['sentence1']); s2=tr(d['sentence2'])
        for a,b,sc in zip(s1,s2,d['score']): rows.append({'split':split,'sentence1':a,'sentence2':b,'score':float(sc)})
    d=pd.DataFrame(rows); d.to_csv(PATHS['bn_stsb_cache'],index=False); del mt; torch.cuda.empty_cache(); return d
bn_df=translate_stsb()
train_df=bn_df[bn_df.split=='train'].reset_index(drop=True)
dev_df  =bn_df[bn_df.split=='validation'].reset_index(drop=True)
test_df =human[['sentence1','sentence2','score']].reset_index(drop=True)
print(f"train={len(train_df)} dev={len(dev_df)} test(human)={len(test_df)}")

README.md:   0%|          | 0.00/5.67k [00:00<?, ?B/s]

train.jsonl.gz:   0%|          | 0.00/278k [00:00<?, ?B/s]

validation.jsonl.gz:   0%|          | 0.00/86.4k [00:00<?, ?B/s]

test.jsonl.gz:   0%|          | 0.00/63.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

...

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Translating train (5749)...
  5749/5749
  5749/5749
Translating validation (1500)...
  1500/1500
  1500/1500
train=5749 dev=1500 test(human)=295


In [ ]:
# ============================================================
# CELL 6 — Normalizer, tokenizer, POS, IDF, weight table
# ============================================================
from transformers import AutoTokenizer
try:
    from normalizer import normalize
    print("✅ Bengali normalizer ready")
except Exception as e:
    def normalize(text, **kwargs): return text
    print("⚠️  Normalizer unavailable -> identity:", e)

tokenizer = AutoTokenizer.from_pretrained(BACKBONE)

try:
    from bnlp import BengaliPOS
    pos_tagger = BengaliPOS(); print("✅ POS tagger ready")
except Exception as e:
    pos_tagger = None; print("⚠️  POS tagger unavailable -> neutral POS:", e)

POS_WEIGHT_TABLE = {'NNP':0.70,'NN':0.50,'NNC':0.50,'NNS':0.48,'VM':0.40,'VAUX':0.15,'JJ':0.35,'JJC':0.33,
    'RB':0.25,'QT':0.30,'PR':0.10,'DEM':0.10,'PSP':0.08,'CC':0.05,'RP':0.08,'NEG':0.12,'XC':0.20,'DEFAULT':0.20}
def pos_weight(t): return POS_WEIGHT_TABLE.get(t, POS_WEIGHT_TABLE['DEFAULT'])

if os.path.exists(PATHS['idf_cache']):
    IDF = pickle.load(open(PATHS['idf_cache'], 'rb')); print(f"✅ IDF loaded ({len(IDF):,})")
else:
    IDF = {}; print("⚠️  No IDF -> IDF=1.0 (POS-only)")

✅ Bengali normalizer ready


config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/528k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

punkt not found. downloading...


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


✅ POS tagger ready
✅ IDF loaded (120,642)


In [ ]:
# ============================================================
# CELL 7 — Encode (per-token POS, IDF, and WORD-ID) + ablatable prior + loaders
#   word-ids enable the morphology-aware (subword->word) pooling head.
# ============================================================
def encode_raw(text):
    tn=normalize(text)
    enc=tokenizer(tn,max_length=MAX_LEN,truncation=True,padding='max_length',
                  return_offsets_mapping=True,return_tensors=None)
    if pos_tagger is not None:
        try: tagged=pos_tagger.tag(tn)
        except Exception: tagged=[(w,'DEFAULT') for w in tn.split()]
    else: tagged=[(w,'DEFAULT') for w in tn.split()]
    posmap={w:pos_weight(t) for w,t in tagged}
    wids=enc.word_ids(); offs=enc['offset_mapping']
    pos=np.zeros(MAX_LEN,np.float32); idf=np.zeros(MAX_LEN,np.float32)
    wid=np.full(MAX_LEN,-1,np.int64)
    for i,w in enumerate(wids):
        if w is not None: wid[i]=w
    for w in set(x for x in wids if x is not None):
        spans=[offs[i] for i,x in enumerate(wids) if x==w]
        surface=tn[min(s for s,e in spans):max(e for s,e in spans)]
        pw=posmap.get(surface,POS_WEIGHT_TABLE['DEFAULT']); iv=IDF.get(surface,1.0)
        for i,x in enumerate(wids):
            if x==w: pos[i]=pw; idf[i]=iv
    mask=np.array(enc['attention_mask'],np.int64)
    return np.array(enc['input_ids'],np.int64), mask, pos, idf, wid

def build_tensors(df):
    I1=[];M1=[];PO1=[];ID1=[];W1=[];I2=[];M2=[];PO2=[];ID2=[];W2=[];Y=[]; n=len(df)
    for k,(_,r) in enumerate(df.iterrows()):
        a=encode_raw(r['sentence1']); b=encode_raw(r['sentence2'])
        I1.append(a[0]);M1.append(a[1]);PO1.append(a[2]);ID1.append(a[3]);W1.append(a[4])
        I2.append(b[0]);M2.append(b[1]);PO2.append(b[2]);ID2.append(b[3]);W2.append(b[4])
        Y.append(float(r['score'])/5.0)
        if (k+1)%500==0: print(f"  {k+1}/{n}",end='\r')
    print()
    t=lambda x,dt: torch.tensor(np.stack(x),dtype=dt)
    return dict(i1=t(I1,torch.long),m1=t(M1,torch.long),pos1=t(PO1,torch.float),idf1=t(ID1,torch.float),wid1=t(W1,torch.long),
                i2=t(I2,torch.long),m2=t(M2,torch.long),pos2=t(PO2,torch.float),idf2=t(ID2,torch.float),wid2=t(W2,torch.long),
                y=torch.tensor(Y, dtype=torch.float))

if os.path.exists(PATHS['tokenized']):
    DATA=torch.load(PATHS['tokenized']); print("✅ cached tensors")
else:
    print("train..."); tr_t=build_tensors(train_df)
    print("dev...");   dv_t=build_tensors(dev_df)
    print("test...");  te_t=build_tensors(test_df)
    DATA={'train':tr_t,'dev':dv_t,'test':te_t}; torch.save(DATA,PATHS['tokenized']); print("✅ cached")

def make_prior(pos, idf, mask, mode):
    if mode=='none': base=torch.zeros_like(pos)
    elif mode=='pos': base=pos
    elif mode=='idf': base=idf
    else: base=pos*idf
    m=mask.float(); cnt=m.sum(1,keepdim=True).clamp(min=1)
    mean=(base*m).sum(1,keepdim=True)/cnt
    var=(((base-mean)*m)**2*m).sum(1,keepdim=True)/cnt
    return ((base-mean)/(var.sqrt()+1e-6))*m

class STSData(Dataset):
    def __init__(self, split, mode):
        d=DATA[split]
        self.i1,self.m1,self.w1=d['i1'],d['m1'],d['wid1']
        self.i2,self.m2,self.w2=d['i2'],d['m2'],d['wid2']
        self.y=d['y']
        self.p1=make_prior(d['pos1'],d['idf1'],d['m1'],mode)
        self.p2=make_prior(d['pos2'],d['idf2'],d['m2'],mode)
    def __len__(self): return len(self.y)
    def __getitem__(self,i):
        return (self.i1[i],self.m1[i],self.p1[i],self.w1[i],
                self.i2[i],self.m2[i],self.p2[i],self.w2[i],self.y[i])

def loaders(mode='posidf', bs=32):
    return (DataLoader(STSData('train',mode),batch_size=bs,shuffle=True),
            DataLoader(STSData('dev',mode),batch_size=64),
            DataLoader(STSData('test',mode),batch_size=64))

train...

dev...
  1500/1500
test...

✅ cached


In [ ]:
# ============================================================
# CELL 8 — Pooling heads (incl. morphology-aware) + bi-encoder
# ============================================================
from transformers import AutoModel

class CLSPooling(nn.Module):
    def forward(self,h,mask,prior=None,wid=None): return h[:,0]

class MeanPooling(nn.Module):
    def forward(self,h,mask,prior=None,wid=None):
        m=mask.unsqueeze(-1).float(); return (h*m).sum(1)/m.sum(1).clamp(min=1e-9)

class AttentionPooling(nn.Module):
    def __init__(self,dim,hidden=256):
        super().__init__(); self.W=nn.Linear(dim,hidden); self.v=nn.Linear(hidden,1,bias=False)
    def forward(self,h,mask,prior=None,wid=None):
        logits=self.v(torch.tanh(self.W(h))).squeeze(-1).masked_fill(mask==0,float('-inf'))
        return (torch.softmax(logits,1).unsqueeze(-1)*h).sum(1)

class PrimedAttentionPooling(nn.Module):   # v1 token-level primed attention (kept as a baseline)
    def __init__(self,dim,hidden=256,lam_init=1.0):
        super().__init__(); self.W=nn.Linear(dim,hidden); self.v=nn.Linear(hidden,1,bias=False)
        self.lam=nn.Parameter(torch.tensor(float(lam_init)))
    def attn_logits(self,h,mask,prior):
        return (self.v(torch.tanh(self.W(h))).squeeze(-1)+self.lam*prior).masked_fill(mask==0,float('-inf'))
    def forward(self,h,mask,prior,wid=None):
        return (torch.softmax(self.attn_logits(h,mask,prior),1).unsqueeze(-1)*h).sum(1)

class MorphPooling(nn.Module):
    '''Morphology-aware pooling: aggregate subword vectors into word vectors
    (counteracting uneven subword fragmentation in inflectional Bengali), then apply
    word-level attention. If primed=True, bias the word-level logits by a per-word
    POS weight (POS is a word property), with a learnable scalar lambda.'''
    def __init__(self,dim,hidden=256,primed=False,lam_init=1.0):
        super().__init__(); self.W=nn.Linear(dim,hidden); self.v=nn.Linear(hidden,1,bias=False)
        self.primed=primed
        if primed: self.lam=nn.Parameter(torch.tensor(float(lam_init)))
    def _word_pool(self,h,mask,wid,prior):
        B,T,H=h.shape; Wn=T
        valid=((wid>=0)&(mask>0))
        widx=wid.clone(); widx[~valid]=Wn                      # dump slot at index Wn
        vf=valid.unsqueeze(-1).float()
        wsum=h.new_zeros(B,Wn+1,H).scatter_add_(1,widx.unsqueeze(-1).expand(-1,-1,H),h*vf)
        wcnt=h.new_zeros(B,Wn+1,1).scatter_add_(1,widx.unsqueeze(-1),vf)
        wvec=wsum/wcnt.clamp(min=1e-9)                          # (B,Wn+1,H)
        wmask=(wcnt.squeeze(-1)>0).float(); wmask[:,Wn]=0.0     # kill dump slot
        wpri=None
        if prior is not None:
            psum=h.new_zeros(B,Wn+1).scatter_add_(1,widx,prior*valid.float())
            wpri=psum/wcnt.squeeze(-1).clamp(min=1e-9)
        return wvec,wmask,wpri
    def attn_logits(self,h,mask,wid,prior):
        wvec,wmask,wpri=self._word_pool(h,mask,wid,prior)
        logits=self.v(torch.tanh(self.W(wvec))).squeeze(-1)
        if self.primed and wpri is not None: logits=logits+self.lam*wpri
        # NaN-safety: a sentence with no valid word tokens (e.g. a degenerate/empty
        # translation that tokenises to special tokens only) leaves an all-masked row;
        # softmax over an all -inf row returns NaN and poisons training on the first
        # such batch. For those rows, re-enable the dump slot (a zero word-vector) so
        # the row is well-defined and the pooled embedding is a finite zero vector.
        empty=(wmask.sum(1)==0)
        if empty.any():
            wmask=wmask.clone(); wmask[empty,-1]=1.0
        neg=torch.finfo(logits.dtype).min
        return wvec,logits.masked_fill(wmask==0,neg)
    def forward(self,h,mask,prior=None,wid=None):
        wvec,logits=self.attn_logits(h,mask,wid,prior)
        return (torch.softmax(logits,1).unsqueeze(-1)*wvec).sum(1)

def make_head(kind,dim):
    if kind=='cls':          return CLSPooling()
    if kind=='mean':         return MeanPooling()
    if kind=='attn':         return AttentionPooling(dim)
    if kind=='primed':       return PrimedAttentionPooling(dim)
    if kind=='morph':        return MorphPooling(dim,primed=False)
    if kind=='morph_primed': return MorphPooling(dim,primed=True)
    raise ValueError(kind)

class BiEncoderSTS(nn.Module):
    def __init__(self,backbone,head):
        super().__init__(); self.enc=AutoModel.from_pretrained(backbone); self.head=head
    def encode(self,ids,mask,prior,wid):
        return self.head(self.enc(input_ids=ids,attention_mask=mask).last_hidden_state,mask,prior,wid)
    def forward(self,i1,m1,p1,w1,i2,m2,p2,w2):
        return F.cosine_similarity(self.encode(i1,m1,p1,w1),self.encode(i2,m2,p2,w2),dim=-1)

HID=AutoModel.from_pretrained(BACKBONE).config.hidden_size

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# # ============================================================
# # CELL 9 — Multi-seed training. Stores per-seed predictions (for consistent CIs).
# # ============================================================
# @torch.no_grad()
# def predict(model,loader):
#     model.eval(); P=[];G=[]
#     for i1,m1,p1,w1,i2,m2,p2,w2,y in loader:
#         i1,m1,p1,w1=i1.to(DEVICE),m1.to(DEVICE),p1.to(DEVICE),w1.to(DEVICE)
#         i2,m2,p2,w2=i2.to(DEVICE),m2.to(DEVICE),p2.to(DEVICE),w2.to(DEVICE)
#         P+=list(model(i1,m1,p1,w1,i2,m2,p2,w2).cpu().numpy()); G+=list(y.numpy())
#     return np.array(G),np.array(P)
# def corr(g,p): return pearsonr(g,p)[0]*100, spearmanr(g,p)[0]*100

# def train_once(kind, mode, seed):
#     set_seed(seed)
#     model=BiEncoderSTS(BACKBONE, make_head(kind,HID)).to(DEVICE)
#     tr,dv,te=loaders(mode); opt=torch.optim.AdamW(model.parameters(),lr=2e-5); lossf=nn.MSELoss()
#     best_sp,best_state=-1,None
#     for ep in range(EPOCHS):
#         model.train()
#         for i1,m1,p1,w1,i2,m2,p2,w2,y in tr:
#             i1,m1,p1,w1=i1.to(DEVICE),m1.to(DEVICE),p1.to(DEVICE),w1.to(DEVICE)
#             i2,m2,p2,w2=i2.to(DEVICE),m2.to(DEVICE),p2.to(DEVICE),w2.to(DEVICE); y=y.to(DEVICE)
#             opt.zero_grad(); lossf(model(i1,m1,p1,w1,i2,m2,p2,w2),y).backward(); opt.step()
#         g,p=predict(model,dv); _,sp=corr(g,p)
#         if sp>best_sp: best_sp=sp; best_state={k:v.cpu() for k,v in model.state_dict().items()}
#     model.load_state_dict(best_state)
#     gT,pT=predict(model,te); prT,spT=corr(gT,pT)
#     return model, prT, spT, pT

# # (kind, prior-mode, label).  morph_primed uses POS-only — the prior that won v1's ablation.
# MAIN = [('cls','none','BanglaBERT + [CLS]'),
#         ('mean','none','BanglaBERT + Mean'),
#         ('attn','none','BanglaBERT + Learned-Attention'),
#         ('primed','posidf','BanglaBERT + Primed-Attention (token)'),
#         ('morph','none','BanglaBERT + Morph-Mean (word)'),
#         ('morph_primed','pos','BanglaBERT + Morph-POS-Attn (word)')]

# ft_metrics={}; ft_preds_seeds={}; ft_models={}
# for kind,mode,label in MAIN:
#     prs=[]; seedpreds=[]
#     for si,seed in enumerate(SEEDS):
#         print(f"[{label}] seed {seed} ...")
#         model,pr,sp,pT=train_once(kind,mode,seed)
#         prs.append((pr,sp)); seedpreds.append(pT)
#         if si==0:
#             ft_models[(kind,mode)]=model
#             torch.save(model.state_dict(), PATHS['ckpt_dir']+f'banglabert_{kind}_{mode}.pt')
#         torch.cuda.empty_cache()
#     ft_metrics[label]=np.array(prs); ft_preds_seeds[label]=seedpreds
#     m=ft_metrics[label].mean(0); s=ft_metrics[label].std(0)
#     print(f"  => {label}: Pearson {m[0]:.2f}±{s[0]:.2f}  Spearman {m[1]:.2f}±{s[1]:.2f}\n")

# # seed-averaged prediction vector per system (used for error analysis & figures, consistently)
# ft_pred_ens={l:np.mean(np.stack(v),0) for l,v in ft_preds_seeds.items()}
# print("✅ multi-seed training done.")


# ============================================================
# CELL 9 — Multi-seed training. Stores per-seed predictions (for consistent CIs).
# ============================================================
import math

@torch.no_grad()
def predict(model,loader):
    model.eval(); P=[];G=[]
    for i1,m1,p1,w1,i2,m2,p2,w2,y in loader:
        i1,m1,p1,w1=i1.to(DEVICE),m1.to(DEVICE),p1.to(DEVICE),w1.to(DEVICE)
        i2,m2,p2,w2=i2.to(DEVICE),m2.to(DEVICE),p2.to(DEVICE),w2.to(DEVICE)
        P+=list(model(i1,m1,p1,w1,i2,m2,p2,w2).cpu().numpy()); G+=list(y.numpy())
    return np.array(G),np.array(P)
def corr(g,p): return pearsonr(g,p)[0]*100, spearmanr(g,p)[0]*100

def train_once(kind, mode, seed):
    set_seed(seed)
    model=BiEncoderSTS(BACKBONE, make_head(kind,HID)).to(DEVICE)
    tr,dv,te=loaders(mode); opt=torch.optim.AdamW(model.parameters(),lr=2e-5); lossf=nn.MSELoss()
    best_sp,best_state=-float('inf'),None
    for ep in range(EPOCHS):
        model.train()
        for i1,m1,p1,w1,i2,m2,p2,w2,y in tr:
            i1,m1,p1,w1=i1.to(DEVICE),m1.to(DEVICE),p1.to(DEVICE),w1.to(DEVICE)
            i2,m2,p2,w2=i2.to(DEVICE),m2.to(DEVICE),p2.to(DEVICE),w2.to(DEVICE); y=y.to(DEVICE)
            opt.zero_grad(); lossf(model(i1,m1,p1,w1,i2,m2,p2,w2),y).backward(); opt.step()
        g,p=predict(model,dv); _,sp=corr(g,p)
        # NaN guard: a NaN validation score must not count as an improvement
        sp_cmp = sp if (sp is not None and not math.isnan(sp)) else -float('inf')
        if sp_cmp>best_sp: best_sp=sp_cmp; best_state={k:v.cpu() for k,v in model.state_dict().items()}
    # fallback: if every epoch was NaN/degenerate, keep final weights instead of crashing
    if best_state is None:
        print(f"  ⚠️  best_state never set (NaN val every epoch?) [kind={kind} mode={mode} seed={seed}] — using final weights")
        best_state={k:v.cpu() for k,v in model.state_dict().items()}
    model.load_state_dict(best_state)
    gT,pT=predict(model,te); prT,spT=corr(gT,pT)
    return model, prT, spT, pT

# (kind, prior-mode, label).  morph_primed uses POS-only — the prior that won v1's ablation.
MAIN = [('cls','none','BanglaBERT + [CLS]'),
        ('mean','none','BanglaBERT + Mean'),
        ('attn','none','BanglaBERT + Learned-Attention'),
        ('primed','posidf','BanglaBERT + Primed-Attention (token)'),
        ('morph','none','BanglaBERT + Morph-Mean (word)'),
        ('morph_primed','pos','BanglaBERT + Morph-POS-Attn (word)')]

ft_metrics={}; ft_preds_seeds={}; ft_models={}
for kind,mode,label in MAIN:
    prs=[]; seedpreds=[]
    for si,seed in enumerate(SEEDS):
        print(f"[{label}] seed {seed} ...")
        model,pr,sp,pT=train_once(kind,mode,seed)
        prs.append((pr,sp)); seedpreds.append(pT)
        if si==0:
            ft_models[(kind,mode)]=model
            torch.save(model.state_dict(), PATHS['ckpt_dir']+f'banglabert_{kind}_{mode}.pt')
        torch.cuda.empty_cache()
    ft_metrics[label]=np.array(prs); ft_preds_seeds[label]=seedpreds
    m=ft_metrics[label].mean(0); s=ft_metrics[label].std(0)
    print(f"  => {label}: Pearson {m[0]:.2f}±{s[0]:.2f}  Spearman {m[1]:.2f}±{s[1]:.2f}\n")

# seed-averaged prediction vector per system (used for error analysis & figures, consistently)
ft_pred_ens={l:np.mean(np.stack(v),0) for l,v in ft_preds_seeds.items()}
print("✅ multi-seed training done.")

[BanglaBERT + [CLS]] seed 42 ...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[BanglaBERT + [CLS]] seed 1 ...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[BanglaBERT + Morph-POS-Attn (word)] seed 2 ...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  => BanglaBERT + Morph-POS-Attn (word): Pearson 76.91±0.69  Spearman 74.17±1.26

✅ multi-seed training done.


In [ ]:
# ============================================================
# CELL 10 — Expanded zero-shot baseline suite (on the human gold set)
# ============================================================
from sentence_transformers import SentenceTransformer
gold_test = DATA['test']['y'].numpy()
ref_preds={}

def st_sim(name, prefix=""):
    m=SentenceTransformer(name,device=DEVICE)
    f=lambda xs:[prefix+x for x in xs]
    e1=m.encode(f(test_df.sentence1.tolist()),convert_to_tensor=True,show_progress_bar=False,normalize_embeddings=True)
    e2=m.encode(f(test_df.sentence2.tolist()),convert_to_tensor=True,show_progress_bar=False,normalize_embeddings=True)
    return (e1*e2).sum(1).cpu().numpy()

def hf_meanpool_sim(name):
    from transformers import AutoTokenizer as AT, AutoModel as AM
    tk=AT.from_pretrained(name); mm=AM.from_pretrained(name).to(DEVICE).eval()
    def emb(texts):
        out=[]
        for i in range(0,len(texts),64):
            enc=tk([normalize(t) for t in texts[i:i+64]],padding=True,truncation=True,max_length=64,return_tensors='pt').to(DEVICE)
            with torch.no_grad(): h=mm(**enc).last_hidden_state
            mask=enc['attention_mask'].unsqueeze(-1).float()
            v=(h*mask).sum(1)/mask.sum(1).clamp(min=1e-9)
            out.append(F.normalize(v,dim=-1).cpu())
        return torch.cat(out)
    e1=emb(test_df.sentence1.tolist()); e2=emb(test_df.sentence2.tolist())
    return (e1*e2).sum(1).numpy()

BASELINES=[
    ("LaBSE (zero-shot)",       lambda: st_sim("sentence-transformers/LaBSE")),
    ("mMPNet (zero-shot)",      lambda: st_sim("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")),
    ("mE5-base (zero-shot)",    lambda: st_sim("intfloat/multilingual-e5-base", prefix="query: ")),
    ("BGE-M3 (zero-shot)",      lambda: st_sim("BAAI/bge-m3")),
    ("MuRIL (zero-shot)",       lambda: hf_meanpool_sim("google/muril-base-cased")),
    ("IndicBERT (zero-shot)",   lambda: hf_meanpool_sim("ai4bharat/indic-bert")),
]
for label,fn in BASELINES:
    try: ref_preds[label]=fn(); print("✅",label)
    except Exception as e: print("⚠️ skip",label,"->",str(e)[:80])

if RUN_LASER:
    try:
        from laserembeddings import Laser
        os.system("python -m laserembeddings download-models")
        laser=Laser()
        e1=laser.embed_sentences([normalize(s) for s in test_df.sentence1],lang='bn')
        e2=laser.embed_sentences([normalize(s) for s in test_df.sentence2],lang='bn')
        e1=e1/np.linalg.norm(e1,axis=1,keepdims=True); e2=e2/np.linalg.norm(e2,axis=1,keepdims=True)
        ref_preds["LASER (zero-shot)"]=np.sum(e1*e2,1); print("✅ LASER")
    except Exception as e: print("⚠️ skip LASER ->",str(e)[:80])

modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ MuRIL (zero-shot)
⚠️ skip IndicBERT (zero-shot) -> You are trying to access a gated repo.
Make sure to have access to it at https:/


In [ ]:
# ============================================================
# CELL 11 — (Optional) FastText cc.bn.300 baseline
# ============================================================
if RUN_FASTTEXT:
    import fasttext, fasttext.util
    fasttext.util.download_model('bn', if_exists='ignore')
    ft=fasttext.load_model('cc.bn.300.bin')
    def ftvec(s):
        toks=normalize(s).split(); vs=[ft.get_word_vector(t) for t in toks] or [np.zeros(300,np.float32)]
        v=np.mean(vs,0); n=np.linalg.norm(v); return v/n if n>0 else v
    e1=np.stack([ftvec(s) for s in test_df.sentence1]); e2=np.stack([ftvec(s) for s in test_df.sentence2])
    ref_preds["FastText cc.bn.300 (mean)"]=np.sum(e1*e2,1); print("✅ FastText baseline computed")
else:
    print("FastText disabled (set RUN_FASTTEXT=True for the SOTA comparison).")

FastText disabled (set RUN_FASTTEXT=True for the SOTA comparison).


In [ ]:
# ============================================================
# CELL 11b — OPTION 1: adapt a strong multilingual encoder to Bengali
#   Light contrastive fine-tuning on translated STS-B, evaluated on the NATIVE gold set.
#   Honest: train is translated, test is native news -> domain shift. If adapted < zero-shot,
#   that is reported plainly and is itself a finding.
# ============================================================
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from torch.utils.data import DataLoader as STLoader

def _ex(df):
    return [InputExample(texts=[str(a),str(b)],label=float(s)/5.0)
            for a,b,s in zip(df.sentence1,df.sentence2,df.score)]
train_ex=_ex(train_df)
dev_eval=EmbeddingSimilarityEvaluator([str(s) for s in dev_df.sentence1],
        [str(s) for s in dev_df.sentence2],[float(s)/5.0 for s in dev_df.score],
        name="bn_dev", show_progress_bar=False)

def _cos(model,df):
    e1=model.encode(df.sentence1.tolist(),convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=False)
    e2=model.encode(df.sentence2.tolist(),convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=False)
    return (e1*e2).sum(1)

adapt_metrics={}; adapt_preds_seeds={}; adapt_zeroshot={}
for tag,name in ADAPT_MODELS.items():
    base=SentenceTransformer(name,device=DEVICE)
    zs=_cos(base,test_df); adapt_zeroshot[tag]=zs
    zsp,zss=corr(gold_test,zs); del base; torch.cuda.empty_cache()
    prs=[]; seedpreds=[]
    for si,seed in enumerate(SEEDS):
        set_seed(seed); print(f"[adapt {tag}] seed {seed} ...")
        model=SentenceTransformer(name,device=DEVICE)
        loss=losses.CosineSimilarityLoss(model)
        loader=STLoader(train_ex,shuffle=True,batch_size=ADAPT_BS)
        warm=int(len(loader)*ADAPT_EPOCHS*0.1)
        try:
            model.fit(train_objectives=[(loader,loss)],evaluator=dev_eval,epochs=ADAPT_EPOCHS,
                      warmup_steps=warm,optimizer_params={'lr':ADAPT_LR},use_amp=True,
                      show_progress_bar=True,save_best_model=False)
        except TypeError:
            # very new sentence-transformers: fall back to a minimal fit signature
            model.fit(train_objectives=[(loader,loss)],epochs=ADAPT_EPOCHS,warmup_steps=warm,show_progress_bar=True)
        p=_cos(model,test_df); prs.append(corr(gold_test,p)); seedpreds.append(p)
        if si==0: model.save(PATHS['ckpt_dir']+f'adapted_{tag}')
        del model; torch.cuda.empty_cache()
    prs=np.array(prs); adapt_metrics[f"{tag} (adapted)"]=prs; adapt_preds_seeds[f"{tag} (adapted)"]=seedpreds
    dP=prs[:,0].mean()-zsp
    print(f"\n=== {tag}: zero-shot {zsp:.2f} -> adapted {prs[:,0].mean():.2f}±{prs[:,0].std():.2f}  "
          f"(Δ Pearson {dP:+.2f}; {'helps' if dP>0 else 'zero-shot wins — report as finding'})\n")
adapt_pred_ens={l:np.mean(np.stack(v),0) for l,v in adapt_preds_seeds.items()}
print("✅ encoder adaptation done.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[adapt LaBSE] seed 42 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Bn Dev Pearson Cosine,Bn Dev Spearman Cosine
180,No log,No log,0.845875,0.844439
360,No log,No log,0.848885,0.847596
500,0.023760,No Log,No Log,No Log
540,0.023760,No log,0.849369,0.848263


...

[adapt LaBSE] seed 2 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Bn Dev Pearson Cosine,Bn Dev Spearman Cosine
180,No log,No log,0.844176,0.842181
360,No log,No log,0.850082,0.848729
500,0.023499,No Log,No Log,No Log
540,0.023499,No log,0.849788,0.848573



=== LaBSE: zero-shot 84.61 -> adapted 90.64±0.15  (Δ Pearson +6.02; helps)

✅ encoder adaptation done.


In [ ]:
# ============================================================
# CELL 12 — Statistics (consistent basis: per-seed, averaged)
# ============================================================
def bootstrap_ci(g,p,n_boot=2000,alpha=0.05,seed=0):
    rng=np.random.default_rng(seed); n=len(g); idx=np.arange(n); rs=[]
    for _ in range(n_boot):
        b=rng.choice(idx,n,replace=True)
        if np.std(g[b])==0 or np.std(p[b])==0: continue
        rs.append(pearsonr(g[b],p[b])[0])
    rs=np.array(rs)*100
    return np.percentile(rs,100*alpha/2), np.percentile(rs,100*(1-alpha/2))

def avg_ci(g, preds_list):
    '''Average the per-seed bootstrap CIs -> a CI that uses all seeds and brackets the seed-mean.'''
    los,his=[],[]
    for p in preds_list:
        lo,hi=bootstrap_ci(g,p); los.append(lo); his.append(hi)
    return float(np.mean(los)), float(np.mean(his))

def williams_test(r_hx,r_hy,r_xy,n):
    detR=1-r_hx**2-r_hy**2-r_xy**2+2*r_hx*r_hy*r_xy; avg=(r_hx+r_hy)/2
    num=(r_hx-r_hy)*np.sqrt((n-1)*(1+r_xy))
    den=np.sqrt(2*((n-1)/(n-3))*detR+(avg**2)*(1-r_xy)**3)
    t=num/den; dfree=n-3; return t,dfree,2*(1-stats.t.cdf(abs(t),dfree))

def williams_multiseed(g, listX, listY, alpha=0.05):
    '''Run Williams per seed (broadcasting a single-vector system across seeds).
    Returns mean ΔP (X-Y), and how many seeds are significant.'''
    K=max(len(listX),len(listY))
    LX=listX*K if len(listX)==1 else listX
    LY=listY*K if len(listY)==1 else listY
    K=min(len(LX),len(LY)); n=len(g); dP=[]; nsig=0
    for k in range(K):
        rx=pearsonr(g,LX[k])[0]; ry=pearsonr(g,LY[k])[0]; rxy=pearsonr(LX[k],LY[k])[0]
        t,df,p=williams_test(rx,ry,rxy,n); dP.append((rx-ry)*100); nsig+=int(p<alpha)
    return float(np.mean(dP)), nsig, K

In [ ]:
# ============================================================
# CELL 13 — Main results table
#   Point estimates = mean±std over seeds.
#   95% CI = average of per-seed bootstrap CIs (fine-tuned/adapted) or single-run bootstrap (zero-shot).
#   "best" is selected by score, not hard-coded.
# ============================================================
rows=[]
for kind,mode,label in MAIN:
    m=ft_metrics[label].mean(0); s=ft_metrics[label].std(0)
    lo,hi=avg_ci(gold_test, ft_preds_seeds[label])
    rows.append({'System':label,'Pearson':round(m[0],2),'Pearson_std':round(s[0],2),
                 'Spearman':round(m[1],2),'95% CI':f'[{lo:.2f}, {hi:.2f}]','type':'fine-tuned (BanglaBERT)'})
for label,prs in adapt_metrics.items():
    m=prs.mean(0); s=prs.std(0); lo,hi=avg_ci(gold_test, adapt_preds_seeds[label])
    rows.append({'System':label,'Pearson':round(m[0],2),'Pearson_std':round(s[0],2),
                 'Spearman':round(m[1],2),'95% CI':f'[{lo:.2f}, {hi:.2f}]','type':'adapted encoder'})
for label,pred in ref_preds.items():
    pr,sp=corr(gold_test,pred); lo,hi=bootstrap_ci(gold_test,pred)
    rows.append({'System':label,'Pearson':round(pr,2),'Pearson_std':np.nan,
                 'Spearman':round(sp,2),'95% CI':f'[{lo:.2f}, {hi:.2f}]','type':'zero-shot'})
res=pd.DataFrame(rows).sort_values('Pearson',ascending=False).reset_index(drop=True)
res.to_csv(PATHS['results_csv'],index=False)
print("HUMAN-ANNOTATED TEST SET (n=%d), seeds=%s"%(len(gold_test),SEEDS))
print("Pearson/Spearman = mean±std over seeds; 95%% CI = avg per-seed bootstrap (zero-shot: single-run bootstrap).\n")
print(res[['System','Pearson','Pearson_std','Spearman','95% CI','type']].to_string(index=False))

# learned lambda for the two primed heads
for k in [('primed','posidf'),('morph_primed','pos')]:
    if k in ft_models and hasattr(ft_models[k].head,'lam'):
        print(f"  learned λ [{k[0]}] = {float(ft_models[k].head.lam.detach().cpu()):.4f}")

best_ft_label=max(ft_metrics, key=lambda l: ft_metrics[l].mean(0)[0])
print(f"\nBest BanglaBERT head (by mean Pearson): {best_ft_label}")

HUMAN-ANNOTATED TEST SET (n=295), seeds=[42, 1, 2]
Pearson/Spearman = mean±std over seeds; 95%% CI = avg per-seed bootstrap (zero-shot: single-run bootstrap).

                               System   Pearson  Pearson_std  Spearman         95% CI                    type
                      LaBSE (adapted) 90.640000         0.15     88.63 [88.48, 92.58]         adapted encoder
                   BGE-M3 (zero-shot) 90.080002          NaN     87.70 [87.59, 92.02]               zero-shot
                    LaBSE (zero-shot) 84.610001          NaN     86.57 [81.56, 87.23]               zero-shot
                 mE5-base (zero-shot) 84.519997          NaN     83.67 [81.00, 87.58]               zero-shot
                   mMPNet (zero-shot) 82.430000          NaN     83.64 [78.68, 85.64]               zero-shot
                    MuRIL (zero-shot) 82.070000          NaN     79.67 [77.77, 85.62]               zero-shot
       BanglaBERT + Morph-Mean (word) 77.580000         0.90     74.40

In [ ]:
# ============================================================
# CELL 14 — Significance (Williams, per-seed: mean ΔP and #significant seeds)
# ============================================================
def show(nameX,Xlist,nameY,Ylist):
    dP,nsig,K=williams_multiseed(gold_test,Xlist,Ylist)
    print(f"{nameX} vs {nameY}: mean ΔP={dP:+.2f}  significant in {nsig}/{K} seeds")

S=ft_preds_seeds
print("=== Morphology-aware pooling ===")
show('Morph-Mean (word)',     S['BanglaBERT + Morph-Mean (word)'],     'Mean',          S['BanglaBERT + Mean'])
show('Morph-POS-Attn (word)', S['BanglaBERT + Morph-POS-Attn (word)'], 'Morph-Mean',    S['BanglaBERT + Morph-Mean (word)'])
show('Morph-POS-Attn (word)', S['BanglaBERT + Morph-POS-Attn (word)'], 'Primed (token)',S['BanglaBERT + Primed-Attention (token)'])
print("\n=== Best fine-tuned / adapted vs strongest zero-shot ===")
best_ft_label=max(ft_metrics, key=lambda l: ft_metrics[l].mean(0)[0])
if 'LaBSE (zero-shot)' in ref_preds:
    show(best_ft_label, S[best_ft_label], 'LaBSE (zero-shot)', [ref_preds['LaBSE (zero-shot)']])
for al,seeds in adapt_preds_seeds.items():
    base_tag=al.split(' ')[0]
    if 'LaBSE (zero-shot)' in ref_preds and base_tag=='LaBSE':
        show(al, seeds, 'LaBSE (zero-shot)', [ref_preds['LaBSE (zero-shot)']])

=== Morphology-aware pooling ===
Morph-Mean (word) vs Mean: mean ΔP=+2.82  significant in 2/3 seeds
Morph-POS-Attn (word) vs Morph-Mean: mean ΔP=-0.67  significant in 0/3 seeds
Morph-POS-Attn (word) vs Primed (token): mean ΔP=+3.63  significant in 2/3 seeds

=== Best fine-tuned / adapted vs strongest zero-shot ===
BanglaBERT + Morph-Mean (word) vs LaBSE (zero-shot): mean ΔP=-7.03  significant in 3/3 seeds
LaBSE (adapted) vs LaBSE (zero-shot): mean ΔP=+6.02  significant in 3/3 seeds


In [17]:
# ============================================================
# CELL 15 — Ablation
#   (a) token-level prior: none / POS / IDF / POS×IDF  (primed head)
#   (b) pooling level: token-attn vs word-attn (morph) vs word-attn+POS (morph_primed)
# ============================================================
abl_metrics={'none':ft_metrics['BanglaBERT + Learned-Attention'],
             'posidf':ft_metrics['BanglaBERT + Primed-Attention (token)']}
abl_lam={'posidf':float(ft_models[('primed','posidf')].head.lam.detach().cpu())}
for mode in ['pos','idf']:
    prs=[]
    for si,seed in enumerate(SEEDS):
        print(f"[ablation primed/{mode}] seed {seed} ...")
        model,pr,sp,pT=train_once('primed',mode,seed); prs.append((pr,sp))
        if si==0: abl_lam[mode]=float(model.head.lam.detach().cpu())
        torch.cuda.empty_cache()
    abl_metrics[mode]=np.array(prs)
order=['none','pos','idf','posidf']; labels={'none':'No prior (=learned attn)','pos':'POS-only','idf':'IDF-only','posidf':'POS×IDF (full)'}
abl_rows=[]
for mode in order:
    mm=abl_metrics[mode].mean(0); ss=abl_metrics[mode].std(0)
    abl_rows.append({'Setting':labels[mode],'Pearson':round(mm[0],2),'Pearson_std':round(ss[0],2),
                     'Spearman':round(mm[1],2),'λ':round(abl_lam.get(mode,float('nan')),3)})
# pooling-level rows
for lab in ['BanglaBERT + Primed-Attention (token)','BanglaBERT + Morph-Mean (word)','BanglaBERT + Morph-POS-Attn (word)']:
    mm=ft_metrics[lab].mean(0); ss=ft_metrics[lab].std(0)
    abl_rows.append({'Setting':lab.replace('BanglaBERT + ',''),'Pearson':round(mm[0],2),
                     'Pearson_std':round(ss[0],2),'Spearman':round(mm[1],2),'λ':np.nan})
abl=pd.DataFrame(abl_rows); abl.to_csv(PATHS['ablation_csv'],index=False)
print("\nABLATION (mean±std over seeds):"); print(abl.to_string(index=False))

[ablation primed/pos] seed 42 ...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[ablation primed/pos] seed 1 ...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[ablation primed/idf] seed 2 ...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



ABLATION (mean±std over seeds):
                 Setting  Pearson  Pearson_std  Spearman     λ
No prior (=learned attn)    76.61         0.44     72.76   NaN
                POS-only    76.00         1.84     72.70 0.999
                IDF-only    75.86         1.94     70.05 0.999
          POS×IDF (full)    73.28         2.19     68.28 0.997
Primed-Attention (token)    73.28         2.19     68.28   NaN
       Morph-Mean (word)    77.58         0.90     74.40   NaN
   Morph-POS-Attn (word)    76.91         0.69     74.17   NaN


In [18]:
# ============================================================
# CELL 16 — Error analysis (best BanglaBERT head, seed-averaged predictions)
# ============================================================
best_label=max(ft_metrics, key=lambda l: ft_metrics[l].mean(0)[0])
best_pred=ft_pred_ens[best_label]
best_pred5=best_pred*5.0; gold5=gold_test*5.0
ea=test_df.copy(); ea['gold']=gold5; ea['pred']=np.round(best_pred5,2); ea['abs_err']=np.abs(best_pred5-gold5)
if 'score_std' in g_raw.columns: ea['annot_disagreement']=g_raw['score_std'].values

bins=pd.cut(gold5,[-.01,1,2,3,4,5.01],labels=['0-1','1-2','2-3','3-4','4-5'])
by_bin=ea.groupby(bins)['abs_err'].agg(['mean','count']).round(3)
print(f"BEST SYSTEM (BanglaBERT head): {best_label}\nMAE by gold-score band:"); print(by_bin.to_string())
if 'annot_disagreement' in ea:
    rdis=pearsonr(ea['annot_disagreement'],ea['abs_err'])[0]
    print(f"\ncorr(model |error|, annotator disagreement) = {rdis:.3f}")
print("\nWorst over-predictions (model >> human):")
print(ea.assign(d=ea['pred']-ea['gold']).sort_values('d',ascending=False)
        [['sentence1','sentence2','gold','pred']].head(5).to_string(index=False))
print("\nWorst under-predictions (model << human):")
print(ea.assign(d=ea['gold']-ea['pred']).sort_values('d',ascending=False)
        [['sentence1','sentence2','gold','pred']].head(5).to_string(index=False))
ea.to_csv(PATHS['erroran_csv'],index=False); print("\n✅ saved",PATHS['erroran_csv'])

BEST SYSTEM (BanglaBERT head): BanglaBERT + Morph-Mean (word)
MAE by gold-score band:
      mean  count
0-1  1.955    165
1-2  1.253     34
2-3  0.796     28
3-4  0.560     26
4-5  0.510     42

corr(model |error|, annotator disagreement) = -0.269

Worst over-predictions (model >> human):
                                                                     sentence1                                                                                      sentence2  gold  pred
                                 ৬ জনকে উদ্ধার করে হাসপাতালে ভর্তি করা হয়েছে।                                                                  ঘটনার তদন্ত শুরু করেছে পুলিশ।   0.0  3.55
                                           ওয়েব ডেস্ক: সাক্ষাত্ অনিবার্য ছিল।                                                                  ওয়েব ডেস্ক : মন্ত্রিত্ব নেই।   0.0  3.46
                                        ওয়েব ডেস্ক : অবশেষে প্রতীক্ষার অবসান।                                                         ওয়েব ডেস্ক: উত্স

In [19]:
# ============================================================
# CELL 17 — Publication figures (300 dpi PNG + vector PDF)
# ============================================================
def save(fig,name):
    for ext in ('png','pdf'): fig.savefig(FIGDIR+name+'.'+ext,bbox_inches='tight')
    plt.close(fig)

# combine all systems with a single, consistent point-estimate/CI basis
all_ens={**ft_pred_ens, **adapt_pred_ens, **ref_preds}
ftype={l:'fine-tuned' for _,_,l in MAIN}
ftype.update({l:'adapted' for l in adapt_pred_ens}); ftype.update({l:'zero-shot' for l in ref_preds})
fr=[]
for label,pred in all_ens.items():
    pr=pearsonr(gold_test,pred)[0]*100; lo,hi=bootstrap_ci(gold_test,pred)
    fr.append({'System':label,'Pearson':pr,'lo':max(pr-lo,0.0),'hi':max(hi-pr,0.0),'type':ftype[label]})
r2=pd.DataFrame(fr).sort_values('Pearson')
cmap={'fine-tuned':'#2e7d32','adapted':'#ad1457','zero-shot':'#1565c0'}
fig,ax=plt.subplots(figsize=(8.5,5.5))
ax.barh(r2['System'],r2['Pearson'],xerr=[r2['lo'].values,r2['hi'].values],
        color=[cmap[t] for t in r2['type']],capsize=3)
ax.set_xlabel('Pearson r × 100 (human gold set)'); ax.set_title('System comparison (point estimate with bootstrap 95% CI)')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=cmap[k],label=k) for k in cmap],loc='lower right')
save(fig,'fig1_results')

fig,ax=plt.subplots(figsize=(7,4))
ax.hist(gold5,bins=np.arange(0,5.5,0.5),color='#6a1b9a',edgecolor='black')
ax.set_xlabel('gold similarity (0–5)'); ax.set_ylabel('# pairs'); ax.set_title(f'Gold-score distribution (n={len(gold5)})')
save(fig,'fig2_distribution')

fig,ax=plt.subplots(figsize=(5.5,5.5))
ax.scatter(gold5,best_pred5,s=14,alpha=0.5,color='#00695c')
m,b=np.polyfit(gold5,best_pred5,1); xs=np.array([0,5]); ax.plot(xs,m*xs+b,'r--',lw=1)
ax.set_xlabel('human gold (0–5)'); ax.set_ylabel('model cosine × 5')
best_r_seedmean = ft_metrics[best_label].mean(0)[0]  # seed-mean Pearson×100 — identical to Table 1
ax.set_title(f'{best_label}\nPearson r = {best_r_seedmean/100:.3f} (seed-mean, matches Table 1)\n'
             f'points = seed-averaged predictions', fontsize=10)
save(fig,'fig3_scatter')

fig,ax=plt.subplots(figsize=(7.5,4.5))
abl_order=['none','pos','idf','posidf']; abl_lab={'none':'No prior','pos':'POS','idf':'IDF','posidf':'POS×IDF'}
ax.bar([abl_lab[m] for m in abl_order],[abl_metrics[m].mean(0)[0] for m in abl_order],
       yerr=[abl_metrics[m].std(0)[0] for m in abl_order],capsize=4,color='#ef6c00')
ax.set_ylabel('Pearson r × 100'); ax.set_title('Token-level linguistic-prior ablation (primed head)')
save(fig,'fig4_ablation')

fig,ax=plt.subplots(figsize=(9,4.5))
labs=[l for _,_,l in MAIN]; data=[ft_metrics[l][:,0] for l in labs]
ax.boxplot(data,labels=[l.replace('BanglaBERT + ','') for l in labs])
for i,d in enumerate(data): ax.scatter([i+1]*len(d),d,color='black',zorder=3,s=18)
ax.set_ylabel('Pearson r × 100'); ax.set_title(f'Across-seed variance (seeds={SEEDS})'); plt.xticks(rotation=20,ha='right')
save(fig,'fig5_seed_variance')

print("✅ figures saved to",FIGDIR,"(png + pdf): fig1_results, fig2_distribution, fig3_scatter, fig4_ablation, fig5_seed_variance")

✅ figures saved to /content/drive/MyDrive/Similarity_V2_Patched/figures/ (png + pdf): fig1_results, fig2_distribution, fig3_scatter, fig4_ablation, fig5_seed_variance


In [20]:
# ============================================================
# CELL 18 — Word-level attention visualization (morphology-aware head), Fig 6
# ============================================================
mp_key=('morph_primed','pos')
if mp_key in ft_models:
    morph=ft_models[mp_key]; morph.eval()
    def word_attn(text):
        ids,mask,pos,idf,wid=encode_raw(text)
        ids_t=torch.tensor(ids[None]).to(DEVICE); mask_t=torch.tensor(mask[None]).to(DEVICE)
        prior=make_prior(torch.tensor(pos[None]),torch.tensor(idf[None]),torch.tensor(mask[None]),'pos').to(DEVICE)
        wid_t=torch.tensor(wid[None]).to(DEVICE)
        with torch.no_grad():
            h=morph.enc(input_ids=ids_t,attention_mask=mask_t).last_hidden_state
            _,logits=morph.head.attn_logits(h,mask_t,wid_t,prior)
            w=torch.softmax(logits,1)[0].cpu().numpy()
        # map each word index -> its surface (first subword span)
        words={}
        toks=tokenizer.convert_ids_to_tokens(ids)
        for i,wi in enumerate(wid):
            if wi>=0 and mask[i]>0 and wi not in words:
                words[wi]=toks[i].replace('##','')
        items=[(words[wi],w[wi]) for wi in sorted(words)]
        return items
    ex=g_raw.sort_values('gold_score',ascending=False).iloc[0]['sentence1']
    items=word_attn(ex)
    fig,ax=plt.subplots(figsize=(min(12,0.5*len(items)+2),3))
    ax.bar(range(len(items)),[w for _,w in items],color='#3949ab')
    ax.set_xticks(range(len(items))); ax.set_xticklabels([t for t,_ in items],rotation=60,ha='right',fontsize=9)
    ax.set_ylabel('word attention weight'); ax.set_title('Morphology-aware word-level attention (POS-primed)')
    for ext in ('png','pdf'): fig.savefig(FIGDIR+'fig6_word_attention.'+ext,bbox_inches='tight')
    plt.close(fig)
    print("✅ fig6_word_attention saved. Example:\n ",ex)
else:
    print("morph_primed head not trained; skipping word-attention figure.")

✅ fig6_word_attention saved. Example:
  ২০১৫ সালে কমপক্ষে ১০০টি মামলা রুজু করে একের পর এক নারীপাচার চক্র ফাঁস হয়।


In [21]:
# ============================================================
# CELL 19 — Datasheet generator
# ============================================================
def wlen(s): return len(str(s).split())
lens=pd.concat([test_df['sentence1'].map(wlen),test_df['sentence2'].map(wlen)])
dist=pd.cut(gold5,[-.01,1,2,3,4,5.01],labels=['0-1','1-2','2-3','3-4','4-5']).value_counts().sort_index()
src_mix = g_raw[['source1','source2']].stack().value_counts().to_dict() if 'source1' in g_raw.columns else "see candidate-pair master"

ds=f'''# Datasheet — Bengali STS evaluation set (news domain)

## Composition
- **Pairs:** {len(test_df)}
- **Language / variety:** Bengali (West-Bengal / Indian news register) — STATE THIS AS A LIMITATION
- **Domain:** news (single-domain)
- **Sentence length (words):** mean {lens.mean():.1f}, median {int(lens.median())}, min {int(lens.min())}, max {int(lens.max())}
- **Gold-score distribution (0–5):** {dist.to_dict()}
- **Source mix:** {src_mix}

## Annotation
- **Annotators:** 3 native Bengali speakers, independent scoring after a calibration round
- **Scale:** SemEval STS 0–5
- **Inter-annotator agreement:** Krippendorff α = 0.863 (interval) / 0.833 (ordinal); ICC(2,k) = 0.950 [0.93, 0.96]; mean pairwise Pearson = 0.885
- **Gold label:** per-pair mean of 3 annotators

## Positioning (literature-checked)
- A prior Bengali STS set exists (Shajalal & Aono, 2018) but is **not publicly available** and has < 1000 pairs; Bengali is absent from SemRel2024 and MUSTS.
- This set is therefore positioned as the **first publicly available, natively sourced, human-annotated Bengali STS *evaluation* set with reported IAA** — comparable in size to the 100-pair Sinhala/Tamil STS test sets admitted to MUSTS.

## Construction
- Candidate sentences mined from native Bengali news (Anandabazar, Zee News); cleaned, normalised (csebuetnlp), exact + MinHash near-dup removed
- Pairs stratified across LaBSE-cosine bins; per-sentence usage capped; annotator-facing order randomised; model similarity hidden

## Recommended use & limitations
- News-domain, Indian-Bengali variety; not general-domain or Bangladeshi-Bengali
- Dissimilar-heavy distribution (~{int(dist.get('0-1',0))}/{len(test_df)} pairs in 0–1)
- Evaluate with Pearson/Spearman; report bootstrap CIs

## Licensing
- Released as sentence pairs + scores (not full articles). Add your license (e.g. CC BY-SA 4.0) and provenance note.
'''
open(PATHS['datasheet'],'w',encoding='utf-8').write(ds)
print(ds); print("✅ datasheet ->",PATHS['datasheet'])

# Datasheet — Bengali STS evaluation set (news domain)

## Composition
- **Pairs:** 295
- **Language / variety:** Bengali (West-Bengal / Indian news register) — STATE THIS AS A LIMITATION
- **Domain:** news (single-domain)
- **Sentence length (words):** mean 9.7, median 9, min 5, max 27
- **Gold-score distribution (0–5):** {'0-1': 165, '1-2': 34, '2-3': 28, '3-4': 26, '4-5': 42}
- **Source mix:** see candidate-pair master

## Annotation
- **Annotators:** 3 native Bengali speakers, independent scoring after a calibration round
- **Scale:** SemEval STS 0–5
- **Inter-annotator agreement:** Krippendorff α = 0.863 (interval) / 0.833 (ordinal); ICC(2,k) = 0.950 [0.93, 0.96]; mean pairwise Pearson = 0.885
- **Gold label:** per-pair mean of 3 annotators

## Positioning (literature-checked)
- A prior Bengali STS set exists (Shajalal & Aono, 2018) but is **not publicly available** and has < 1000 pairs; Bengali is absent from SemRel2024 and MUSTS.
- This set is therefore positioned as the **first

## What remains — manual steps the notebook cannot do

The cells above produce every result, statistic and figure the paper needs. What is left is scholarship:

1. **Framing (do NOT claim "first Bengali STS dataset").** Use the literature-checked claim: *first
   publicly available, natively sourced, human-annotated Bengali STS evaluation set with reported
   IAA.* Cite Shajalal & Aono (2018) (exists but not public, < 1000 pairs), the BanglaBERT paper
   (reports it is not public), MUSTS (excludes it for size; admits 100-pair Sinhala/Tamil sets), and
   SemRel2024 (Bengali absent). The ready-to-paste related-work paragraph + BibTeX is in the
   `related_work_and_contribution.tex` file delivered alongside this notebook.

2. **Reading the modelling results honestly.**
   - If **Morph-POS-Attn (word) > Primed-Attention (token)** and/or **> Mean**, you have a positive,
     novel finding: morphology-aware pooling helps in inflectional Bengali, and POS helps at the word
     level where it belongs. Lead the modelling section with it.
   - If it ties/loses, report it plainly — combined with the v1 finding (a token-level POS×IDF prior
     does not help; POS-only edges the hybrid), this is a clean, publishable analysis of *where*
     linguistic priors do and do not help fine-tuned Bengali encoders.
   - If **LaBSE (adapted) > LaBSE (zero-shot)** on the native set, that is your concrete "best system."
     If not, the honest story is that strong multilingual encoders are already near-ceiling on this
     set and translated-train adaptation does not transfer to native news.

3. **Writing.** Related work establishing the gap; datasheet prose; limitations (Indian-Bengali
   variety, news-only, dissimilar-heavy, 295 pairs as an *evaluation* set); ethics/consent.